# spglib

## Example 1: space groups of textbook structures

In [ ]:
# Example 1: get_spacegroup finds the space group of BCC iron, rock salt and diamond
import numpy as np
import spglib

bcc = (np.eye(3) * 2.87, [[0, 0, 0], [0.5, 0.5, 0.5]], [26, 26])
rocksalt = (np.eye(3) * 5.64, [[0, 0, 0], [0, 0.5, 0.5], [0.5, 0, 0.5], [0.5, 0.5, 0],
                               [0.5, 0, 0], [0, 0.5, 0], [0, 0, 0.5], [0.5, 0.5, 0.5]], [11] * 4 + [17] * 4)
fcc = [[0, 0, 0], [0, 0.5, 0.5], [0.5, 0, 0.5], [0.5, 0.5, 0]]
diamond = (np.eye(3) * 3.567, fcc + [[x + 0.25, y + 0.25, z + 0.25] for x, y, z in fcc], [6] * 8)
groups = [spglib.get_spacegroup(cell) for cell in (bcc, rocksalt, diamond)]
assert groups == ["Im-3m (229)", "Fm-3m (225)", "Fd-3m (227)"], groups
print("Example 1:", groups, "spglib", spglib.spg_get_version_full())

## Example 2: the symmetry dataset

In [ ]:
# Example 2: get_symmetry_dataset reports operations, Wyckoff letters and the international symbol
import numpy as np
import spglib

rutile = (np.diag([4.59, 4.59, 2.96]), [[0, 0, 0], [0.5, 0.5, 0.5], [0.305, 0.305, 0], [0.695, 0.695, 0],
                                         [0.805, 0.195, 0.5], [0.195, 0.805, 0.5]], [22, 22, 8, 8, 8, 8])
dataset = spglib.get_symmetry_dataset(rutile, symprec=1e-3)
assert dataset.international == "P4_2/mnm" and dataset.number == 136, (dataset.international, dataset.number)
assert len(dataset.rotations) == 16 and list(dataset.wyckoffs) == ["a", "a", "f", "f", "f", "f"], dataset.wyckoffs
print("Example 2:", dataset.international, len(dataset.rotations), "operations", list(dataset.wyckoffs))

## Example 3: primitive and standardised cells

In [ ]:
# Example 3: find_primitive shrinks the conventional FCC cell to one atom; standardize_cell restores it
import numpy as np
import spglib

fcc = (np.eye(3) * 4.05, [[0, 0, 0], [0, 0.5, 0.5], [0.5, 0, 0.5], [0.5, 0.5, 0]], [13] * 4)
lattice, positions, numbers = spglib.find_primitive(fcc)
assert len(numbers) == 1 and np.isclose(abs(np.linalg.det(lattice)), 4.05 ** 3 / 4), np.linalg.det(lattice)
std_lattice, std_positions, std_numbers = spglib.standardize_cell((lattice, positions, numbers), to_primitive=False)
assert len(std_numbers) == 4 and np.allclose(np.diag(std_lattice), 4.05), std_lattice
print("Example 3: primitive volume", round(abs(np.linalg.det(lattice)), 3), "-> conventional", len(std_numbers), "atoms")

## Example 4: irreducible k-points

In [ ]:
# Example 4: get_ir_reciprocal_mesh folds an 8x8x8 mesh by the cubic symmetry
import numpy as np
import spglib

simple_cubic = (np.eye(3) * 3.0, [[0, 0, 0]], [1])
mapping, grid = spglib.get_ir_reciprocal_mesh([8, 8, 8], simple_cubic, is_shift=[0, 0, 0])
irreducible = np.unique(mapping)
assert grid.shape == (512, 3) and len(irreducible) == 35, len(irreducible)  # 512 points fold to 35 under Oh
print("Example 4:", len(grid), "k-points ->", len(irreducible), "irreducible")

## Example 5: Niggli reduction and the space-group database

In [ ]:
# Example 5: niggli_reduce gives the reduced basis of a skewed cell; the database lists Hall symbols and types
import numpy as np
import spglib

skewed = np.array([[3.0, 0.0, 0.0], [5.0, 4.0, 0.0], [1.0, 2.0, 5.0]])
reduced = spglib.niggli_reduce(skewed)
assert np.isclose(abs(np.linalg.det(reduced)), abs(np.linalg.det(skewed)))  # same lattice, same volume
assert np.linalg.norm(reduced, axis=1).max() < np.linalg.norm(skewed, axis=1).max()
info = spglib.get_spacegroup_type(523)  # Hall number 523 is Fm-3m
assert info.international_short == "Fm-3m" and info.number == 225, info
print("Example 5:", np.round(np.linalg.norm(reduced, axis=1), 3), info.hall_symbol)